# Odometry Exercise 4 — plotting the square benchmark

Load repeated clockwise and anticlockwise square trials, plot the paths reported by local odometry, enter the independently measured start and end poses, and compare physical closure with the odometry estimate.

This notebook is an introduction to plotting odometry data. Start with the supplied synthetic example so that you can see what each cell produces. The example is not evidence about your robot. When you are ready, change the settings in the **Use your own data** cell and run the notebook again.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
rng = np.random.default_rng(2026)


## 1. Use the example or your own data

Leave `USE_EXAMPLE_DATA` set to `True` on your first run. To use your measurements, upload the CSV received over Wi-Fi, set `USE_EXAMPLE_DATA = False`, and enter its filename. This is the main cell you need to edit.

The notebook expects one row per odometry sample with these columns: `direction`, `trial_num`, `sample_time_ms`, `odometry_x_mm`, `odometry_y_mm`, and `odometry_theta_rad`. Use `clockwise` and `anticlockwise` consistently in the direction column.


In [ ]:
USE_EXAMPLE_DATA = True
CSV_FILENAME = "odometry_exercise04_square_trials.csv"

print("Using:", "synthetic example" if USE_EXAMPLE_DATA else CSV_FILENAME)


## 2. Create the synthetic square trials

Run this cell when using the example. You do not need to understand or edit the generation code. The lines it creates are local-odometry estimates, not measured physical paths or expected results for your robot.


In [ ]:
example_rows = []
side_length_mm = 300
example_conditions = [
    ("clockwise", 1, 3.0, -1.0, 0.020),
    ("clockwise", 2, 4.0, -2.0, 0.025),
    ("clockwise", 3, 2.0, -1.5, 0.018),
    ("clockwise", 4, 5.0, -2.5, 0.030),
    ("clockwise", 5, 3.5, -1.8, 0.022),
    ("anticlockwise", 1, -2.0, 1.0, -0.015),
    ("anticlockwise", 2, -3.0, 1.5, -0.020),
    ("anticlockwise", 3, -1.5, 2.0, -0.012),
    ("anticlockwise", 4, -4.0, 1.2, -0.026),
    ("anticlockwise", 5, -2.5, 1.8, -0.018),
]

for direction, trial_num, drift_x, drift_y, drift_theta in example_conditions:
    turn_sign = -1 if direction == "clockwise" else 1
    waypoints = [
        (0, 0, 0),
        (side_length_mm, 0, 0),
        (side_length_mm, 0, turn_sign * np.pi / 2),
        (side_length_mm, turn_sign * side_length_mm, turn_sign * np.pi / 2),
        (side_length_mm, turn_sign * side_length_mm, turn_sign * np.pi),
        (0, turn_sign * side_length_mm, turn_sign * np.pi),
        (0, turn_sign * side_length_mm, turn_sign * 3 * np.pi / 2),
        (0, 0, turn_sign * 3 * np.pi / 2),
        (0, 0, turn_sign * 2 * np.pi),
    ]
    sample_num = 0
    for segment_num in range(8):
        start = np.array(waypoints[segment_num], dtype=float)
        finish = np.array(waypoints[segment_num + 1], dtype=float)
        for fraction in np.linspace(0, 1, 8, endpoint=False):
            pose = start + fraction * (finish - start)
            total_progress = (segment_num + fraction) / 8
            example_rows.append({
                "direction": direction,
                "trial_num": trial_num,
                "sample_time_ms": 10_000 * trial_num + 50 * sample_num,
                "odometry_x_mm": pose[0] + drift_x * total_progress,
                "odometry_y_mm": pose[1] + drift_y * total_progress,
                "odometry_theta_rad": pose[2] + drift_theta * total_progress,
            })
            sample_num += 1

    final_pose = np.array(waypoints[-1], dtype=float)
    example_rows.append({
        "direction": direction,
        "trial_num": trial_num,
        "sample_time_ms": 10_000 * trial_num + 50 * sample_num,
        "odometry_x_mm": final_pose[0] + drift_x,
        "odometry_y_mm": final_pose[1] + drift_y,
        "odometry_theta_rad": final_pose[2] + drift_theta,
    })

example_data = pd.DataFrame(example_rows)


## 3. Load and preview the selected data

When `USE_EXAMPLE_DATA` is false, `pd.read_csv(...)` reads your file. The final line displays its first five rows. Check that the direction, trial number, time and pose columns look as you intended.


In [ ]:
if USE_EXAMPLE_DATA:
    data = example_data.copy()
else:
    data = pd.read_csv(CSV_FILENAME)

data.head()


## 4. Prepare readable time and trial labels

Each trial may begin at a different StampC3 timestamp. Subtracting the first timestamp makes every trial begin at zero seconds. The trial label keeps clockwise and anticlockwise trials distinct even when they use the same number.


In [ ]:
data = data.sort_values(["direction", "trial_num", "sample_time_ms"])
data["elapsed_s"] = (
    data["sample_time_ms"]
    - data.groupby(["direction", "trial_num"])["sample_time_ms"].transform("first")
) / 1000
data["heading_deg"] = np.degrees(data["odometry_theta_rad"])
data["trial_label"] = (
    data["direction"].str.title() + " " + data["trial_num"].astype(str)
)

data[["direction", "trial_num", "elapsed_s", "heading_deg"]].head()


## 5. Plot the paths reported by local odometry

These lines show the model-estimated paths from the StampC3. They are not measurements of the physical paths followed by the robot. The two panels make it easier to compare clockwise and anticlockwise trials.


In [ ]:
grid = sns.relplot(
    data=data,
    x="odometry_x_mm",
    y="odometry_y_mm",
    hue="trial_label",
    col="direction",
    kind="line",
    estimator=None,
    height=5,
    aspect=1,
)
grid.set_axis_labels("Odometry x (mm)", "Odometry y (mm)")
grid.set_titles("{col_name}")
grid.figure.suptitle("Square paths reported by local odometry", y=1.03)
for axis in grid.axes.flat:
    axis.set_aspect("equal", adjustable="box")
    axis.axhline(0, color="grey", linewidth=0.8, linestyle="--")
    axis.axvline(0, color="grey", linewidth=0.8, linestyle="--")
plt.show()


## 6. Plot the reported heading through each trial

The heading plot shows how the local model accumulated the four turns. Clockwise and anticlockwise trials use opposite signs.


In [ ]:
grid = sns.relplot(
    data=data,
    x="elapsed_s",
    y="heading_deg",
    hue="trial_label",
    col="direction",
    kind="line",
    estimator=None,
    height=4,
    aspect=1.2,
)
grid.set_axis_labels("Elapsed time (s)", "Odometry heading (degrees)")
grid.set_titles("{col_name}")
grid.figure.suptitle("Heading reported by local odometry", y=1.03)
plt.show()


## 7. Extract odometry closure from each log

The following cell creates one row per trial. It subtracts the first reported pose from the final reported pose. The heading result is wrapped into the interval from −180 to 180 degrees.


In [ ]:
trial_summary = (
    data.groupby(["direction", "trial_num"], as_index=False)
    .agg(
        odometry_initial_x_mm=("odometry_x_mm", "first"),
        odometry_initial_y_mm=("odometry_y_mm", "first"),
        odometry_initial_heading_rad=("odometry_theta_rad", "first"),
        odometry_final_x_mm=("odometry_x_mm", "last"),
        odometry_final_y_mm=("odometry_y_mm", "last"),
        odometry_final_heading_rad=("odometry_theta_rad", "last"),
    )
)
trial_summary["odometry_closure_x_mm"] = (
    trial_summary["odometry_final_x_mm"]
    - trial_summary["odometry_initial_x_mm"]
)
trial_summary["odometry_closure_y_mm"] = (
    trial_summary["odometry_final_y_mm"]
    - trial_summary["odometry_initial_y_mm"]
)
odometry_heading_change_deg = np.degrees(
    trial_summary["odometry_final_heading_rad"]
    - trial_summary["odometry_initial_heading_rad"]
)
trial_summary["odometry_closure_heading_deg"] = (
    odometry_heading_change_deg + 180
) % 360 - 180

trial_summary


## 8. Enter the independently measured physical poses

The odometry log cannot provide the physical start and end poses. Enter those observations in this cell. When using your own data, replace the example rows so that `direction` and `trial_num` match your log. Enter headings in degrees using the same sign convention as the odometry. The `np.nan` entries in the own-data template mean that no measurement has been entered yet; replace them with your observations. Also record the side length, surface and frozen model geometry used for the trial. Do not replace the measured final pose with the ideal value of zero.


In [ ]:
if USE_EXAMPLE_DATA:
    physical_measurements = pd.DataFrame({
        "direction": ["clockwise"] * 5 + ["anticlockwise"] * 5,
        "trial_num": [1, 2, 3, 4, 5, 1, 2, 3, 4, 5],
        "square_side_mm": [300] * 10,
        "surface": ["example surface"] * 10,
        "model_wheel_radius_mm": [16.0] * 10,
        "model_wheel_separation_mm": [90.0] * 10,
        "physical_initial_x_mm": [0.5, -0.5, 0.0, 0.8, -0.4, 0.2, -0.6, 0.4, -0.3, 0.7],
        "physical_initial_y_mm": [0.0, 0.5, -0.5, -0.2, 0.3, -0.4, 0.2, 0.5, -0.5, 0.0],
        "physical_initial_heading_deg": [0.2, -0.3, 0.0, 0.4, -0.2, 0.1, -0.4, 0.3, -0.2, 0.0],
        "physical_final_x_mm": [15.0, 12.5, 16.0, 13.5, 14.5, -10.5, -13.0, -9.0, -12.0, -11.0],
        "physical_final_y_mm": [-9.0, -12.0, -8.0, -11.0, -10.0, 9.5, 12.0, 10.0, 13.0, 11.0],
        "physical_final_heading_deg": [3.2, 2.2, 3.5, 2.8, 3.0, -2.4, -3.0, -1.8, -2.7, -2.2],
    })
else:
    physical_measurements = pd.DataFrame({
        "direction": ["clockwise", "anticlockwise"],
        "trial_num": [1, 1],
        "square_side_mm": [np.nan, np.nan],
        "surface": ["", ""],
        "model_wheel_radius_mm": [np.nan, np.nan],
        "model_wheel_separation_mm": [np.nan, np.nan],
        "physical_initial_x_mm": [np.nan, np.nan],
        "physical_initial_y_mm": [np.nan, np.nan],
        "physical_initial_heading_deg": [np.nan, np.nan],
        "physical_final_x_mm": [np.nan, np.nan],
        "physical_final_y_mm": [np.nan, np.nan],
        "physical_final_heading_deg": [np.nan, np.nan],
    })

physical_measurements


## 9. Calculate physical closure and endpoint residuals

The physical closure is the measured final pose minus the measured initial pose. Each residual is physical closure minus odometry closure, matching the convention used in Exercise 4.


In [ ]:
comparison = trial_summary.merge(
    physical_measurements,
    on=["direction", "trial_num"],
)
comparison["physical_closure_x_mm"] = (
    comparison["physical_final_x_mm"]
    - comparison["physical_initial_x_mm"]
)
comparison["physical_closure_y_mm"] = (
    comparison["physical_final_y_mm"]
    - comparison["physical_initial_y_mm"]
)
physical_heading_change_deg = (
    comparison["physical_final_heading_deg"]
    - comparison["physical_initial_heading_deg"]
)
comparison["physical_closure_heading_deg"] = (
    physical_heading_change_deg + 180
) % 360 - 180

comparison["residual_x_mm"] = (
    comparison["physical_closure_x_mm"]
    - comparison["odometry_closure_x_mm"]
)
comparison["residual_y_mm"] = (
    comparison["physical_closure_y_mm"]
    - comparison["odometry_closure_y_mm"]
)
comparison["residual_heading_deg"] = (
    comparison["physical_closure_heading_deg"]
    - comparison["odometry_closure_heading_deg"]
    + 180
) % 360 - 180

comparison[[
    "direction", "trial_num",
    "physical_closure_x_mm", "physical_closure_y_mm",
    "physical_closure_heading_deg",
    "odometry_closure_x_mm", "odometry_closure_y_mm",
    "odometry_closure_heading_deg",
    "residual_x_mm", "residual_y_mm", "residual_heading_deg",
]]


## 10. Plot the measured physical closure points

Each small point is an independently measured return position relative to its measured start. The table gives each direction group's mean and standard deviation, and the large × marks its mean position. These points do not imply that the physical path between the start and end was measured.


In [ ]:
closure_summary = (
    comparison.groupby("direction", as_index=False)
    .agg(
        mean_closure_x_mm=("physical_closure_x_mm", "mean"),
        spread_closure_x_mm=("physical_closure_x_mm", "std"),
        mean_closure_y_mm=("physical_closure_y_mm", "mean"),
        spread_closure_y_mm=("physical_closure_y_mm", "std"),
    )
)
display(closure_summary)

fig, ax = plt.subplots(figsize=(7, 6))
sns.scatterplot(
    data=comparison,
    x="physical_closure_x_mm",
    y="physical_closure_y_mm",
    hue="direction",
    style="direction",
    s=90,
    ax=ax,
)
sns.scatterplot(
    data=closure_summary,
    x="mean_closure_x_mm",
    y="mean_closure_y_mm",
    hue="direction",
    marker="X",
    s=260,
    legend=False,
    ax=ax,
)
ax.scatter(0, 0, color="black", marker="+", s=160, label="ideal closure")
ax.axhline(0, color="grey", linewidth=0.8, linestyle="--")
ax.axvline(0, color="grey", linewidth=0.8, linestyle="--")
ax.set_aspect("equal", adjustable="box")
ax.set(
    title="Independently measured physical closure",
    xlabel="Physical closure x (mm)",
    ylabel="Physical closure y (mm)",
)
plt.show()


## 11. Plot position residuals

A residual of zero means that the physical closure and the odometry closure agree for that component. Keep x and y in millimetres and inspect the clockwise and anticlockwise groups separately.


In [ ]:
position_residuals = comparison.melt(
    id_vars=["direction", "trial_num"],
    value_vars=["residual_x_mm", "residual_y_mm"],
    var_name="component",
    value_name="residual_mm",
)
position_residuals["trial_label"] = (
    position_residuals["direction"].str.title()
    + " "
    + position_residuals["trial_num"].astype(str)
)

fig, ax = plt.subplots(figsize=(10, 5))
sns.scatterplot(
    data=position_residuals,
    x="trial_label",
    y="residual_mm",
    hue="direction",
    style="component",
    s=90,
    ax=ax,
)
ax.axhline(0, color="black", linestyle="--")
ax.set(
    title="Physical closure minus odometry closure",
    xlabel="Trial",
    ylabel="Position residual (mm)",
)
ax.tick_params(axis="x", rotation=35)
plt.tight_layout()
plt.show()


## 12. Plot heading residuals

Heading is plotted separately because degrees and millimetres should not be combined into one unexplained score.


In [ ]:
comparison["trial_label"] = (
    comparison["direction"].str.title()
    + " "
    + comparison["trial_num"].astype(str)
)

fig, ax = plt.subplots(figsize=(10, 4.5))
sns.scatterplot(
    data=comparison,
    x="trial_label",
    y="residual_heading_deg",
    hue="direction",
    style="direction",
    s=90,
    ax=ax,
)
ax.axhline(0, color="black", linestyle="--")
ax.set(
    title="Physical closure minus odometry closure",
    xlabel="Trial",
    ylabel="Heading residual (degrees)",
)
ax.tick_params(axis="x", rotation=35)
plt.tight_layout()
plt.show()


## What to notice

- Do repeated physical closure points form different clockwise and anticlockwise groups?
- How large is the spread within each direction compared with the distance between the two group centres?
- Does local odometry return close to its starting pose even when the physical robot does not?
- Do the residuals keep the same sign within either direction?

Save the notebook with your plots. The odometry trajectories are model estimates; only the independently measured closure points describe the physical return positions.
